# Construction du Calendrier des Événements - Paris 2024

**Projet :** Smart Mobility Paris - Prédiction NO2 sur le périphérique  
**Sortie :** `evenements_paris_2024.csv` (1 ligne par jour, avec indicateurs binaires)

---

### Objectif

Construire une table journalière 2024 contenant les variables événementielles qui influencent le trafic et donc la pollution sur le périphérique :

- **Jours fériés** (11 jours fériés nationaux 2024)
- **Vacances scolaires** Zone C (Paris)
- **Jeux Olympiques et Paralympiques** Paris 2024 (impact massif sur la circulation)
- **Indicateurs temporels** (jour de la semaine, weekend)

Ces variables seront jointes aux données météo, pollution et validations IDFM pour entraîner le modèle XGBoost.

### Sources

- Jours fériés : Code du travail français (11 jours fériés légaux)
- Vacances scolaires : Arrêté du 7 décembre 2022 - Ministère de l'Éducation Nationale
- JO Paris 2024 : 26 juillet - 11 août 2024
- JOP Paris 2024 : 28 août - 8 septembre 2024

## 0. Imports et configuration

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Chemins relatifs depuis notebooks/
OUTPUT_DIR = "../data/proceessed"

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

## 1. Création du squelette : 366 jours de 2024

2024 est une **année bissextile** → 366 jours.

In [ ]:
df = pd.DataFrame({
    "DATE": pd.date_range(start="2024-01-01", end="2024-12-31", freq="D")
})

print(f"Nombre de jours : {len(df)}")
print(f"Période : {df['DATE'].min().date()} -> {df['DATE'].max().date()}")
df.head()

## 2. Variables temporelles de base

In [ ]:
df["JOUR_SEMAINE"]     = df["DATE"].dt.dayofweek          # 0=lundi, 6=dimanche
df["NOM_JOUR"]         = df["DATE"].dt.day_name(locale="fr_FR.utf8") if False else df["DATE"].dt.day_name()
df["WEEKEND"]          = (df["JOUR_SEMAINE"] >= 5).astype(int)
df["MOIS"]             = df["DATE"].dt.month
df["NUM_SEMAINE"]      = df["DATE"].dt.isocalendar().week.astype(int)
df["JOUR_ANNEE"]       = df["DATE"].dt.dayofyear

df.head(10)

## 3. Jours fériés 2024 (11 jours fériés nationaux)

Source : Code du travail français.

| # | Date | Nom |
|---|---|---|
| 1 | 2024-01-01 | Jour de l'An |
| 2 | 2024-04-01 | Lundi de Pâques |
| 3 | 2024-05-01 | Fête du Travail |
| 4 | 2024-05-08 | Victoire 1945 |
| 5 | 2024-05-09 | Ascension |
| 6 | 2024-05-20 | Lundi de Pentecôte |
| 7 | 2024-07-14 | Fête nationale |
| 8 | 2024-08-15 | Assomption |
| 9 | 2024-11-01 | Toussaint |
| 10 | 2024-11-11 | Armistice 1918 |
| 11 | 2024-12-25 | Noël |

In [ ]:
jours_feries_2024 = {
    "2024-01-01": "Jour de l'An",
    "2024-04-01": "Lundi de Pâques",
    "2024-05-01": "Fête du Travail",
    "2024-05-08": "Victoire 1945",
    "2024-05-09": "Ascension",
    "2024-05-20": "Lundi de Pentecôte",
    "2024-07-14": "Fête nationale",
    "2024-08-15": "Assomption",
    "2024-11-01": "Toussaint",
    "2024-11-11": "Armistice 1918",
    "2024-12-25": "Noël",
}

# Conversion en datetime pour le merge
feries_df = pd.DataFrame({
    "DATE": pd.to_datetime(list(jours_feries_2024.keys())),
    "NOM_FERIE": list(jours_feries_2024.values())
})

df = df.merge(feries_df, on="DATE", how="left")
df["JOUR_FERIE"] = df["NOM_FERIE"].notna().astype(int)

print(f"Nombre de jours fériés : {df['JOUR_FERIE'].sum()}")
df[df["JOUR_FERIE"] == 1][["DATE", "NOM_JOUR", "NOM_FERIE"]]

## 4. Vacances scolaires Zone C (Paris) - 2024

Source : Arrêté du 7 décembre 2022 - Ministère de l'Éducation Nationale.

**Note :** la Zone C inclut Paris, Créteil, Versailles, Montpellier et Toulouse. Les vacances commencent le **soir du dernier jour de classe** et finissent **le matin du jour de reprise**, mais pour simplifier on considère que le jour de fin est aussi en vacances (le lundi de reprise est exclu).

| Période | Début | Fin |
|---|---|---|
| Hiver 2024 | sam 10 fév | dim 25 fév |
| Printemps 2024 | sam 6 avr | dim 21 avr |
| Pont Ascension 2024 | mer 8 mai (soir) | dim 12 mai |
| Été 2024 | sam 6 juil | dim 1er sept |
| Toussaint 2024 | sam 19 oct | dim 3 nov |
| Noël 2024-2025 | sam 21 déc 2024 | dim 5 jan 2025 |

In [ ]:
# Liste de tuples (date_debut, date_fin, type)
vacances_2024 = [
    ("2024-01-01", "2024-01-07", "Noël 2023-2024"),     # fin des vacances de Noël précédentes
    ("2024-02-10", "2024-02-25", "Hiver"),
    ("2024-04-06", "2024-04-21", "Printemps"),
    ("2024-05-08", "2024-05-12", "Pont Ascension"),
    ("2024-07-06", "2024-09-01", "Été"),
    ("2024-10-19", "2024-11-03", "Toussaint"),
    ("2024-12-21", "2024-12-31", "Noël 2024-2025"),     # début des vacances de Noël (suite en 2025)
]

# Initialisation des colonnes
df["VACANCES_SCOLAIRES"] = 0
df["TYPE_VACANCES"] = None

for date_debut, date_fin, type_vac in vacances_2024:
    mask = (df["DATE"] >= date_debut) & (df["DATE"] <= date_fin)
    df.loc[mask, "VACANCES_SCOLAIRES"] = 1
    df.loc[mask, "TYPE_VACANCES"] = type_vac

print(f"Nombre de jours en vacances scolaires : {df['VACANCES_SCOLAIRES'].sum()}")
print(f"\nRépartition par période :")
print(df[df["VACANCES_SCOLAIRES"] == 1]["TYPE_VACANCES"].value_counts())

## 5. Jeux Olympiques et Paralympiques Paris 2024 ⭐

**C'est l'événement le plus important pour le projet** : les JO ont radicalement modifié la circulation parisienne avec :
- Restrictions de circulation autour des sites olympiques
- **Voies olympiques réservées sur le périphérique** (impact direct sur ton sujet !)
- Vague de départ des Parisiens en vacances anticipées
- Affluence touristique massive

**Période détaillée :**
- 🔥 **JO** : 26 juillet - 11 août 2024 (cérémonie d'ouverture sur la Seine)
- ⏸️ Entre-deux : 12 août - 27 août 2024
- 🔥 **JOP (Paralympiques)** : 28 août - 8 septembre 2024

In [ ]:
# Période JO
mask_jo = (df["DATE"] >= "2024-07-26") & (df["DATE"] <= "2024-08-11")
df["JO"] = mask_jo.astype(int)

# Période JOP (Paralympiques)
mask_jop = (df["DATE"] >= "2024-08-28") & (df["DATE"] <= "2024-09-08")
df["JOP"] = mask_jop.astype(int)

# Variable agrégée "événement olympique en cours"
df["EVENEMENT_OLYMPIQUE"] = (df["JO"] | df["JOP"]).astype(int)

print(f"Jours de JO : {df['JO'].sum()}")
print(f"Jours de JOP : {df['JOP'].sum()}")
print(f"Total jours olympiques : {df['EVENEMENT_OLYMPIQUE'].sum()}")

## 6. Variables agrégées utiles pour le modèle

In [ ]:
# Jour "non ouvré" : weekend OU férié → la circulation pendulaire est très différente
df["JOUR_NON_OUVRE"] = ((df["WEEKEND"] == 1) | (df["JOUR_FERIE"] == 1)).astype(int)

# "Jour avec circulation perturbée" : férié OU vacances OU JO/JOP
df["JOUR_PERTURBE"] = (
    (df["JOUR_FERIE"] == 1) |
    (df["VACANCES_SCOLAIRES"] == 1) |
    (df["EVENEMENT_OLYMPIQUE"] == 1)
).astype(int)

print(f"Jours non ouvrés (weekend + fériés) : {df['JOUR_NON_OUVRE'].sum()}")
print(f"Jours avec circulation perturbée : {df['JOUR_PERTURBE'].sum()}")

## 7. Vérifications

In [ ]:
# Vue d'ensemble
print("--- Aperçu ---")
print(df.head(10))

print("\n--- Statistiques par variable ---")
for col in ["WEEKEND", "JOUR_FERIE", "VACANCES_SCOLAIRES", "JO", "JOP", 
            "EVENEMENT_OLYMPIQUE", "JOUR_NON_OUVRE", "JOUR_PERTURBE"]:
    print(f"  {col:25s} : {df[col].sum():3d} jours ({df[col].mean()*100:.1f}%)")

In [ ]:
# Vérification croisée : les jours fériés doivent tous être marqués
print("Vérification : tous les jours fériés sont bien identifiés")
df[df["JOUR_FERIE"] == 1][["DATE", "NOM_JOUR", "NOM_FERIE"]].reset_index(drop=True)

## 8. Visualisation : carte calendaire des événements

In [ ]:
# Construction d'un score d'événement pour visualisation
df["SCORE_EVENEMENT"] = (
    df["WEEKEND"] * 1 +
    df["VACANCES_SCOLAIRES"] * 2 +
    df["JOUR_FERIE"] * 3 +
    df["EVENEMENT_OLYMPIQUE"] * 4
)

# Heatmap calendaire
df_pivot = df.copy()
df_pivot["SEMAINE"] = df_pivot["DATE"].dt.isocalendar().week
df_pivot["JOUR"] = df_pivot["DATE"].dt.dayofweek

heatmap_data = df_pivot.pivot_table(
    index="JOUR", columns="SEMAINE", values="SCORE_EVENEMENT", aggfunc="max"
)

fig, ax = plt.subplots(figsize=(20, 4))
sns.heatmap(
    heatmap_data, cmap="YlOrRd", cbar_kws={"label": "Score événement"},
    yticklabels=["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"], ax=ax,
)
ax.set_title("Calendrier 2024 - Score d'événement (rouge = JO/JOP, orange = férié, jaune = vacances/weekend)")
ax.set_xlabel("Semaine de l'année")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_calendrier_evenements.png", dpi=120)
plt.show()

df = df.drop(columns=["SCORE_EVENEMENT"])

In [ ]:
# Répartition mensuelle des événements
monthly_events = df.groupby("MOIS").agg({
    "JOUR_FERIE": "sum",
    "VACANCES_SCOLAIRES": "sum",
    "EVENEMENT_OLYMPIQUE": "sum",
    "WEEKEND": "sum",
})

fig, ax = plt.subplots(figsize=(13, 5))
monthly_events.plot(kind="bar", stacked=False, ax=ax, color=["#e74c3c", "#f39c12", "#9b59b6", "#95a5a6"])
ax.set_title("Répartition mensuelle des événements - 2024")
ax.set_xlabel("Mois")
ax.set_ylabel("Nombre de jours")
ax.legend(title="Type d'événement")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/06_evenements_mensuels.png", dpi=120)
plt.show()

## 9. Export du fichier final

In [ ]:
# Sélection et ordre final des colonnes
colonnes_finales = [
    "DATE",
    "JOUR_SEMAINE", "NOM_JOUR", "WEEKEND",
    "MOIS", "NUM_SEMAINE", "JOUR_ANNEE",
    "JOUR_FERIE", "NOM_FERIE",
    "VACANCES_SCOLAIRES", "TYPE_VACANCES",
    "JO", "JOP", "EVENEMENT_OLYMPIQUE",
    "JOUR_NON_OUVRE", "JOUR_PERTURBE",
]
df_final = df[colonnes_finales].copy()

output_csv = f"{OUTPUT_DIR}/evenements_paris_2024.csv"
df_final.to_csv(output_csv, index=False)

print(f"Fichier sauvegardé : {output_csv}")
print(f"Dimensions : {df_final.shape[0]} lignes x {df_final.shape[1]} colonnes")
print(f"\nColonnes finales :")
for c in df_final.columns:
    print(f"  - {c}")

---

## Bilan

| Variable | Description | Utilité pour le modèle |
|---|---|---|
| `WEEKEND` | 1 si samedi/dimanche | Trafic pendulaire absent |
| `JOUR_FERIE` | 1 si jour férié | Trafic réduit |
| `VACANCES_SCOLAIRES` | 1 si vacances Zone C | Trafic réduit (Parisiens partent) |
| `JO` | 1 si pendant les JO | ⭐ **Restrictions de circulation, voies olympiques sur le périph** |
| `JOP` | 1 si pendant les Paralympiques | ⭐ Mêmes effets, plus modérés |
| `EVENEMENT_OLYMPIQUE` | 1 si JO ou JOP | Variable agrégée |
| `JOUR_NON_OUVRE` | weekend OU férié | Variable agrégée |
| `JOUR_PERTURBE` | tout type d'événement | Variable agrégée "super-feature" |

**Prochaine étape** : joindre ce fichier avec :
- `meteo_paris_2024_clean.csv`
- les validations IDFM agrégées par jour
- les données pollution Airparif

→ Dataset final pour XGBoost.